In [ ]:

# =====================================================
# MACHINE LEARNING: PREDIKSI PERFORMA SISWA
# Dataset: Student Performance Factors (Kaggle)
# Algoritma: Random Forest Regressor
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. MEMUAT DATASET
df = pd.read_csv("./DatasetStudentPerformanceFactors.csv")

print("Ukuran dataset:", df.shape)
print("\nContoh data:")
print(df.head())

print("\nJumlah data kosong:")
print(df.isnull().sum())

# 2. PEMBERSIHAN DATA
df = df.drop_duplicates()
df = df.dropna(subset=["Exam_Score"])

# 3. MENENTUKAN FITUR DAN TARGET
# Exam_Score adalah nilai yang akan diprediksi.
X = df.drop(columns=["Exam_Score"])
y = df["Exam_Score"]

# Memisahkan kolom numerik dan kategorikal
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# 4. PREPROCESSING DATA
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# 5. MEMBAGI DATA TRAINING DAN TESTING
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nData training:", X_train.shape[0])
print("Data testing:", X_test.shape[0])

# 6. MEMBUAT MODEL
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

# 7. MELATIH MODEL
model.fit(X_train, y_train)
print("\nModel berhasil dilatih!")

# 8. MELAKUKAN PREDIKSI DATA TESTING
y_pred = model.predict(X_test)

# 9. EVALUASI MODEL
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== HASIL EVALUASI MODEL =====")
print(f"MAE  : {mae:.2f} poin")
print(f"RMSE : {rmse:.2f} poin")
print(f"R²   : {r2:.3f}")

# 10. VISUALISASI NILAI AKTUAL VS PREDIKSI
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.6)

batas_min = min(y_test.min(), y_pred.min())
batas_max = max(y_test.max(), y_pred.max())

plt.plot(
    [batas_min, batas_max],
    [batas_min, batas_max],
    linestyle="--"
)

plt.xlabel("Nilai Aktual")
plt.ylabel("Nilai Prediksi")
plt.title("Nilai Aktual vs Prediksi Performa Siswa")
plt.tight_layout()
plt.show()

# 11. ANALISIS FITUR YANG PALING BERPENGARUH
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = model.named_steps[
    "regressor"
].feature_importances_

feature_importance = pd.Series(
    importances,
    index=feature_names
).sort_values(ascending=False)

print("\n===== 10 FITUR TERPENTING =====")
print(feature_importance.head(10))

feature_importance.head(10).sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

plt.title("Fitur Terpenting dalam Model")
plt.xlabel("Feature Importance")
plt.tight_layout()
plt.show()

# 12. MEMPREDIKSI DATA SISWA BARU
# Gunakan nilai contoh untuk fitur yang tersedia
siswa_baru = X.iloc[[0]].copy()

# Contoh perubahan input siswa baru
siswa_baru.loc[:, "Hours_Studied"] = 20
siswa_baru.loc[:, "Attendance"] = 90
siswa_baru.loc[:, "Previous_Scores"] = 80

prediksi = model.predict(siswa_baru)[0]

print("\n===== PREDIKSI SISWA BARU =====")
print("Jam belajar per minggu : 20")
print("Kehadiran (%)          : 90")
print("Nilai sebelumnya       : 80")
print(f"Prediksi nilai ujian   : {prediksi:.2f}")
